# Runner — Executa la pipeline per a tots els models

Aquest notebook executa `forecasting_pipeline.ipynb` per a cada model definit a `MODELS`,
injectant el `MODEL_PATH` corresponent via **papermill**.

Els notebooks executats es guarden a `outputs/` i els resultats a `../results/<MODEL_NAME>/`.

**Prerequisits:**
```
pip install papermill statsmodels arch pmdarima scikit-learn torch
```

In [2]:
from pathlib import Path
import time

try:
    import papermill as pm
except ImportError:
    raise ImportError("Instal·la papermill: pip install papermill")

# ── VERSIÓ ───────────────────────────────────────────────────────────────────
# "v1" → models/models_v1  (fix cobertura, sense millores addicionals)
# "v2" → models/           (millores v2 en curs)
MODEL_VERSION = "v2"

MODELS_DIR = Path(f"../models/{MODEL_VERSION}")
# ── MODELS A EXECUTAR ────────────────────────────────────────────────────────
MODELS = [
    str(MODELS_DIR / "arima.py"),
    str(MODELS_DIR / "gru.py"),
    str(MODELS_DIR / "random_forest.py"),
    str(MODELS_DIR / "garch.py"),
    str(MODELS_DIR / "ets.py"),
    str(MODELS_DIR / "drift.py"),
    str(MODELS_DIR / "naive.py"),
    str(MODELS_DIR / "rnn.py"),
    str(MODELS_DIR / "xgboost_model.py"),
]
# Comenta la línia del model si vols saltar-lo (ex. si torch no està instal·lat)
# ─────────────────────────────────────────────────────────────────────────────

PIPELINE_NB = "forecasting_pipeline.ipynb"
OUTPUTS_DIR = Path("outputs") / MODEL_VERSION
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Versió: {MODEL_VERSION}  |  Models dir: {MODELS_DIR}")
print(f"Models a executar: {len(MODELS)}")
for m in MODELS:
    print(f"  {Path(m).stem}")

Versió: v2  |  Models dir: ../models/v2
Models a executar: 9
  arima
  gru
  random_forest
  garch
  ets
  drift
  naive
  rnn
  xgboost_model


In [3]:
results = []   # (model_name, status, elapsed_s, error)

for model_path in MODELS:
    model_name  = Path(model_path).stem
    output_path = OUTPUTS_DIR / f"{model_name}_output.ipynb"

    print(f"\n{'='*60}")
    print(f" Executant: {model_name}  [{MODEL_VERSION}]")
    print(f"{'='*60}")

    t0 = time.time()
    try:
        pm.execute_notebook(
            PIPELINE_NB,
            str(output_path),
            parameters={"MODEL_PATH": model_path, "RESULTS_VERSION": MODEL_VERSION},
            kernel_name="python3",
            progress_bar=False,
        )
        elapsed = time.time() - t0
        results.append((model_name, "OK", round(elapsed, 1), ""))
        print(f" Completat en {elapsed:.1f}s")

    except Exception as e:
        elapsed = time.time() - t0
        results.append((model_name, "ERROR", round(elapsed, 1), str(e)[:120]))
        print(f" ERROR: {e}")


 Executant: arima  [v2]


/home/joanguest/Documents/GitHub Repositories/TimeSeriesExercice/venv/lib/python3.12/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


 Completat en 47.2s

 Executant: gru  [v2]
 Completat en 1417.7s

 Executant: random_forest  [v2]
 Completat en 31.5s

 Executant: garch  [v2]
 Completat en 6.8s

 Executant: ets  [v2]
 Completat en 6.9s

 Executant: drift  [v2]
 Completat en 5.2s

 Executant: naive  [v2]
 Completat en 5.1s

 Executant: rnn  [v2]
 Completat en 1094.9s

 Executant: xgboost_model  [v2]
 Completat en 39.5s


In [4]:
import pandas as pd

df = pd.DataFrame(results, columns=["model", "status", "temps_s", "error"])
print(df.to_string(index=False))

ok    = (df["status"] == "OK").sum()
total = len(df)
print(f"\nResultat: {ok}/{total} models executats correctament")

        model status  temps_s error
        arima     OK     47.2      
          gru     OK   1417.7      
random_forest     OK     31.5      
        garch     OK      6.8      
          ets     OK      6.9      
        drift     OK      5.2      
        naive     OK      5.1      
          rnn     OK   1094.9      
xgboost_model     OK     39.5      

Resultat: 9/9 models executats correctament
